# Bare ONNX models

[Back to the introduction](ch01_demo.ipynb). Run in order. `OnnxModel` runs one `.onnx` file that has no Hugging Face repository around it: no `config.json`, no tokenizer. NumPy goes in, NumPy comes out, and the pre- and post-processing is plain Python.

This notebook runs [UltraFace](https://github.com/Linzaer/Ultra-Light-Fast-Generic-Face-Detector-1MB) (`version-RFB-320`, MIT licence, 1.2 MB), a face detector that Transformers.js cannot load as a pipeline. It looks for faces in a random photo from Wikimedia Commons' "Featured pictures of people". Each photo has its own licence, which the notebook prints with the author. Pillow decodes and draws the photo, so a native Python needs `pip install 'lmlite[native,notebooks]'`. Use `backend="webgpu"` where the browser has it.

In [ ]:
import io
import json
import random
import re
import urllib.parse
import urllib.request

import numpy as np
from PIL import Image, ImageDraw

try:
    import pyodide_http

    pyodide_http.patch_all()  # urllib goes through the browser's fetch
except ImportError:
    pass

from lmlite import OnnxModel

MODEL_URL = (
    "https://raw.githubusercontent.com/Linzaer/Ultra-Light-Fast-Generic-Face-Detector-1MB/master/models/onnx/"
    "version-RFB-320.onnx"
)


def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "lmlite-notebook"})
    with urllib.request.urlopen(request) as response:
        return response.read()

## Load the model

The browser downloads the file and keeps it in the Cache API, so loading it again makes no request. `inputs` and `outputs` describe what the model takes and returns.

In [ ]:
model = await OnnxModel.load(MODEL_URL, backend="wasm", licence="MIT")
print(model.inputs)
print(model.outputs)
info = model.model_info()
print(f"{info['size']} bytes on {info['backend']} ({info['execution_provider']}), cache: {info['cache_status']}")

## Pre- and post-processing in NumPy

UltraFace takes one RGB image resized to 320 x 240, scaled to about [-1, 1], as `(1, 3, 240, 320)` floats. It returns 4,420 candidate boxes with a face score each, and the boxes overlap: keep the confident ones, then drop the ones that overlap a better box (non-maximum suppression).

In [ ]:
def preprocess(image):
    pixels = np.asarray(image.convert("RGB").resize((320, 240)), dtype=np.float32)
    return ((pixels - 127) / 128).transpose(2, 0, 1)[None]


def overlap_ratio(box, boxes):
    """Intersection over union of one box with many; boxes are (x1, y1, x2, y2)."""
    inner = np.clip(np.minimum(box[2:], boxes[:, 2:]) - np.maximum(box[:2], boxes[:, :2]), 0, None).prod(axis=1)
    area = lambda b: (b[..., 2:] - b[..., :2]).prod(axis=-1)
    return inner / (area(box) + area(boxes) - inner)


def non_max_suppression(boxes, scores, max_overlap=0.3):
    order = np.argsort(scores)[::-1]
    kept = []
    while len(order):
        best, order = order[0], order[1:]
        kept.append(best)
        if len(order):
            order = order[overlap_ratio(boxes[best], boxes[order]) <= max_overlap]
    return kept


async def detect_faces(image, threshold=0.7):
    """Boxes (x1, y1, x2, y2 as fractions of the image) and scores of the faces in a PIL image."""
    outputs = await model.run({"input": preprocess(image)})
    scores, boxes = outputs["scores"][0, :, 1], outputs["boxes"][0]
    candidates = np.flatnonzero(scores > threshold)
    kept = non_max_suppression(boxes[candidates], scores[candidates])
    return [(boxes[candidates[i]], float(scores[candidates[i]])) for i in kept]

## A random photo

The Wikimedia Commons API lists the first 50 files of the category with their licences and thumbnails. The notebook tries random ones until it finds a face, because some featured pictures show none.

In [ ]:
COMMONS = "https://commons.wikimedia.org/w/api.php?" + urllib.parse.urlencode(
    {
        "action": "query",
        "format": "json",
        "origin": "*",
        "generator": "categorymembers",
        "gcmtitle": "Category:Featured pictures of people",
        "gcmtype": "file",
        "gcmlimit": 50,
        "prop": "imageinfo",
        "iiprop": "url|extmetadata",
        "iiurlwidth": 640,
        "iiextmetadatafilter": "LicenseShortName|Artist",
    }
)
photos = list(json.loads(fetch(COMMONS))["query"]["pages"].values())

for photo in random.sample(photos, min(8, len(photos))):
    details = photo["imageinfo"][0]
    image = Image.open(io.BytesIO(fetch(details["thumburl"]))).convert("RGB")
    faces = await detect_faces(image)
    if faces:
        break

Draw the boxes, and credit the photographer.

In [ ]:
draw = ImageDraw.Draw(image)
scale = [image.width, image.height] * 2  # fractions of the image to pixels
for box, score in faces:
    draw.rectangle((box * scale).tolist(), outline="red", width=3)

meta = details["extmetadata"]
author = re.sub("<[^>]+>", "", meta.get("Artist", {}).get("value", "unknown author"))
print(f"{len(faces)} face(s) in {photo['title']}, scores {[round(score, 2) for _, score in faces]}")
print(f"Photo: {author}, {meta.get('LicenseShortName', {}).get('value', 'see the file page')} (Wikimedia Commons)")
image

Release the model; the cached file stays for the next run.

In [ ]:
await model.dispose()